In [1]:
!pip install -q --upgrade pip
!pip install -q \
  pandas numpy tqdm rapidfuzz scikit-learn networkx datasets \
  transformers==4.44.2 accelerate==0.33.0 bitsandbytes==0.45.5 \
  torch regex tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
cupy

In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import random
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import json
import re
import os
import itertools
from rapidfuzz import fuzz

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASET_ID = "willcb/wdc-products-multi"

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

MAX_ATTRS_PER_SOURCE_FOR_LLM = 35
MAX_LLM_LINKAGE_CALLS = 300
MAX_LLM_FUSION_CALLS = 50

MATCH_THRESHOLD = 0.40
BORDERLINE_LOW = 0.35
BORDERLINE_HIGH = 0.50

PROJECT_DIR = Path("/content/llm_bdi_project")
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
PROMPTS_DIR = PROJECT_DIR / "prompts"

for d in [PROJECT_DIR, DATA_DIR, RAW_DIR, PROCESSED_DIR, RESULTS_DIR, PROMPTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASET_ID = "willcb/wdc-products-multi"

print("Loading dataset:", DATASET_ID)
ds = load_dataset(DATASET_ID, split="train")
df_raw = ds.to_pandas()

print("Raw shape:", df_raw.shape)
print(df_raw.columns)
df_raw.head()

Loading dataset: willcb/wdc-products-multi


README.md:   0%|          | 0.00/631 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.28M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10422 [00:00<?, ? examples/s]

Raw shape: (10422, 7)
Index(['id', 'brand', 'title', 'description', 'price', 'priceCurrency',
       'cluster_id'],
      dtype='object')


,id,brand,title,description,price,priceCurrency,cluster_id
0,60546106,https://www.leadersystems.com.au/Images/MECS3-...,Crucial 8GB (1x8GB) DDR3L UDIMM 1600MHz ECC Un...,Crucial 8GB (1x8GB) DDR3L UDIMM 1600MHz ECC Un...,201.0,AUD,1026272
1,5297625,None,ASUS GeForce RTX 2080 SUPER DUAL EVO OC,None,0,SEK,3154731
2,54382056,None,Šiltovka New Era Clean Trucker Chicago Bulls Cap,None,2.59E1,EUR,46857262
3,44008793,None,ASUS PRIME X570-PRO scheda madre Presa AM4 ATX...,"ASUS PRIME X570-PRO, AMD, Presa AM4, AMD Ryzen...",289.49,EUR,2848082
4,24382278,None,Swiss Military Hanowa Undercover 06-4307.30.007,06-430730007,4936.00,CZK,4246715


In [2]:
required_cols = ["id", "brand", "title", "description", "price", "priceCurrency", "cluster_id"]
missing = [c for c in required_cols if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

df = df_raw[required_cols].copy()

df["cluster_id"] = df["cluster_id"].astype(str)
df["id"] = df["id"].astype(str)

df = df.dropna(subset=["title", "cluster_id"])
df["title"] = df["title"].astype(str)

cluster_sizes = df.groupby("cluster_id").size()
eligible_clusters = cluster_sizes[cluster_sizes >= 2].sort_values(ascending=False).index.tolist()

print("Eligible clusters >=2 records:", len(eligible_clusters))

selected_clusters = []
n_records = 0

for cid in eligible_clusters:
    selected_clusters.append(cid)
    n_records += min(cluster_sizes[cid], 6)
    if n_records >= 1800 and len(selected_clusters) >= 300:
        break

df = df[df["cluster_id"].isin(selected_clusters)].copy()

df = (
    df.sort_values(["cluster_id", "id"])
      .groupby("cluster_id")
      .head(6)
      .reset_index(drop=True)
)

print("Working records:", len(df))
print("Working entities:", df["cluster_id"].nunique())

SOURCE_NAMES = ["source_alpha", "source_beta", "source_gamma"]

df["within_cluster_idx"] = df.groupby("cluster_id").cumcount()
df["source"] = df["within_cluster_idx"].apply(lambda x: SOURCE_NAMES[x % 3])

df["record_id"] = [f"r{i}" for i in range(len(df))]
df["spec_id"] = df["source"] + "//" + df["id"].astype(str)
df["entity_id"] = df["cluster_id"].astype(str)

print(df[["record_id", "spec_id", "source", "entity_id", "title"]].head())

Eligible clusters >=2 records: 1600
Working records: 1800
Working entities: 300
  record_id                 spec_id        source entity_id  \
0        r0  source_alpha//27282600  source_alpha   1001446   
1        r1    source_beta//3658626   source_beta   1001446   
2        r2  source_gamma//45955817  source_gamma   1001446   
3        r3  source_alpha//58093698  source_alpha   1001446   
4        r4    source_beta//6750484   source_beta   1001446   

                                               title  
0  Apple Lightning Digital AV Adapter - Lightning...  
1              Lightning to Digital AV Adapter White  
2                 Apple Lightning Digital AV Adaptor  
3                 Apple Lightning Digital AV Adapter  
4              Apple Lightning to Digital AV Adapter  


In [3]:
MEDIATED_SCHEMA = [
    "title",
    "brand",
    "description",
    "price",
    "priceCurrency"
]

SOURCE_SCHEMAS = {
    "source_alpha": {
        "title": "product_name",
        "brand": "manufacturer",
        "description": "details",
        "price": "cost",
        "priceCurrency": "currency"
    },
    "source_beta": {
        "title": "name",
        "brand": "brand_name",
        "description": "description_text",
        "price": "amount",
        "priceCurrency": "currency_code"
    },
    "source_gamma": {
        "title": "item_title",
        "brand": "maker",
        "description": "specifications",
        "price": "listed_price",
        "priceCurrency": "money"
    }
}

def safe_value(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def row_to_attrs(row):
    source = row["source"]
    mapping = SOURCE_SCHEMAS[source]
    attrs = {}

    for mediated_attr, source_attr in mapping.items():
        value = safe_value(row[mediated_attr])
        if value:
            attrs[source_attr] = value

    return attrs

records = []
for _, row in df.iterrows():
    records.append({
        "record_id": row["record_id"],
        "spec_id": row["spec_id"],
        "source": row["source"],
        "entity_id": row["entity_id"],
        "attrs": row_to_attrs(row)
    })

work_records = pd.DataFrame(records)

print("work_records:", work_records.shape)
print("sources:", work_records["source"].nunique())
print("entities:", work_records["entity_id"].nunique())

schema_rows = []
for source, mapping in SOURCE_SCHEMAS.items():
    for mediated_attr, source_attr in mapping.items():
        schema_rows.append({
            "source": source,
            "attribute": source_attr,
            "source_attribute_id": f"{source}//{source_attr}",
            "target_attribute_name": mediated_attr
        })

schema_gt = pd.DataFrame(schema_rows)

def build_attribute_profile(records_df):
    rows = []
    for _, row in records_df.iterrows():
        source = row["source"]
        attrs = row["attrs"]
        for k, v in attrs.items():
            source_attribute_id = f"{source}//{k}"
            rows.append({
                "source": source,
                "attribute": k,
                "source_attribute_id": source_attribute_id,
                "value": str(v)
            })
    return pd.DataFrame(rows)

attr_profile = build_attribute_profile(work_records)

attr_stats = (
    attr_profile
    .groupby(["source", "attribute", "source_attribute_id"])
    .agg(
        count=("value", "count"),
        samples=("value", lambda x: list(pd.Series(x).dropna().astype(str).head(5)))
    )
    .reset_index()
)

schema_eval_gt = schema_gt.copy()

work_records.to_pickle(PROCESSED_DIR / "work_records.pkl")
schema_gt.to_csv(PROCESSED_DIR / "schema_ground_truth.csv", index=False)
attr_stats.to_csv(PROCESSED_DIR / "attribute_stats.csv", index=False)
pd.Series(MEDIATED_SCHEMA, name="mediated_attribute").to_csv(PROCESSED_DIR / "mediated_schema.csv", index=False)

print("Mediated schema:", MEDIATED_SCHEMA)
print("Schema GT:")
display(schema_gt)

print("Attribute stats:")
display(attr_stats)

work_records: (1800, 5)
sources: 3
entities: 300
Mediated schema: ['title', 'brand', 'description', 'price', 'priceCurrency']
Schema GT:


,source,attribute,source_attribute_id,target_attribute_name
0,source_alpha,product_name,source_alpha//product_name,title
1,source_alpha,manufacturer,source_alpha//manufacturer,brand
2,source_alpha,details,source_alpha//details,description
3,source_alpha,cost,source_alpha//cost,price
4,source_alpha,currency,source_alpha//currency,priceCurrency
5,source_beta,name,source_beta//name,title
6,source_beta,brand_name,source_beta//brand_name,brand
7,source_beta,description_text,source_beta//description_text,description
8,source_beta,amount,source_beta//amount,price
9,source_beta,currency_code,source_beta//currency_code,priceCurrency


Attribute stats:


,source,attribute,source_attribute_id,count,samples
0,source_alpha,cost,source_alpha//cost,563,"[65, 75, 46.99, 6.9E1, 102.62]"
1,source_alpha,currency,source_alpha//currency,548,"[CAD, AUD, USD, SGD, GBP]"
2,source_alpha,details,source_alpha//details,430,[Use the Lightning Digital AV Adapter with you...
3,source_alpha,manufacturer,source_alpha//manufacturer,191,"[Shimano, Coolermaster, SAMSUNG ELECTRONICS GM..."
4,source_alpha,product_name,source_alpha//product_name,600,[Apple Lightning Digital AV Adapter - Lightnin...
5,source_beta,amount,source_beta//amount,551,"[99.99, 49.00, 63.99, 29.74, 0.0]"
6,source_beta,brand_name,source_beta//brand_name,207,"[Apple, Cooler Master, ASUS, ASUS, Samsung]"
7,source_beta,currency_code,source_beta//currency_code,531,"[NZD, USD, AUD, GBP, PLN]"
8,source_beta,description_text,source_beta//description_text,432,[Use the Lightning Digital AV Adapter with you...
9,source_beta,name,source_beta//name,600,"[Lightning to Digital AV Adapter White, Apple ..."


In [4]:
def count_gold_pairs(records_df):
    total = 0
    rid_to_source = dict(zip(records_df["record_id"], records_df["source"]))

    for _, g in records_df.groupby("entity_id"):
        ids = sorted(g["record_id"].tolist())

        for a, b in itertools.combinations(ids, 2):
            if rid_to_source[a] != rid_to_source[b]:
                total += 1

    return total

def count_conflicts(records_df, mediated_schema, source_schemas):
    conflicts = 0

    for entity_id, group in records_df.groupby("entity_id"):
        for mediated_attr in mediated_schema:
            values = []

            for _, row in group.iterrows():
                source = row["source"]
                source_attr = source_schemas[source][mediated_attr]
                val = row["attrs"].get(source_attr, "")
                val = str(val).strip().lower()

                if val:
                    values.append(val)

            if len(set(values)) > 1:
                conflicts += 1

    return conflicts

n_sources = work_records["source"].nunique()
n_records = len(work_records)
n_entities = work_records["entity_id"].nunique()
n_positive_pairs = count_gold_pairs(work_records)
n_conflicts = count_conflicts(work_records, MEDIATED_SCHEMA, SOURCE_SCHEMAS)

print("Dataset requirement check")
print("Sources:", n_sources)
print("Records:", n_records)
print("Integrated entities:", n_entities)
print("Known positive matching pairs:", n_positive_pairs)
print("Conflicting attribute values:", n_conflicts)
print("Mediated attributes:", len(MEDIATED_SCHEMA))

if n_sources < 3:
    print("WARNING: < 3 sources")
if n_records < 1000:
    print("WARNING: < 1000 records")
if n_entities < 200:
    print("WARNING: < 200 entities")
if n_positive_pairs < 300:
    print("WARNING: < 300 positive pairs")
if n_conflicts < 100:
    print("WARNING: < 100 conflicts")
if len(MEDIATED_SCHEMA) < 5:
    print("WARNING: < 5 mediated attributes")

Dataset requirement check
Sources: 3
Records: 1800
Integrated entities: 300
Known positive matching pairs: 3600
Conflicting attribute values: 1220
Mediated attributes: 5


In [5]:
def normalize_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    x = str(x).lower()
    x = re.sub(r"\s+", " ", x)
    x = x.strip()
    return x

def normalize_key(x):
    x = normalize_text(x)
    x = x.replace("_", " ")
    x = x.replace("-", " ")
    x = re.sub(r"[^a-z0-9 ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_value(x):
    x = normalize_text(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def tokenize(x):
    return [t for t in normalize_key(x).split() if t]

def first_meaningful_token(text):
    stop = {"the", "a", "an", "and", "or", "for", "with", "monitor", "lcd", "led", "full", "hd"}
    for t in tokenize(text):
        if len(t) > 2 and t not in stop:
            return t
    return ""

def extract_number(x):
    x = normalize_text(x)
    m = re.search(r"\d+(?:\.\d+)?", x)
    if m:
        try:
            return float(m.group(0))
        except Exception:
            return None
    return None

def sim_text(a, b):
    a = normalize_text(a)
    b = normalize_text(b)
    if not a or not b:
        return 0.0
    return fuzz.token_set_ratio(a, b) / 100.0

def exact_or_sim(a, b):
    a = normalize_text(a)
    b = normalize_text(b)
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    return sim_text(a, b)

In [6]:
def baseline_schema_align(attr_stats, mediated_schema, threshold=62):
    target_norm = {t: normalize_key(t) for t in mediated_schema}
    rows = []

    for _, r in attr_stats.iterrows():
        source_attr_id = r["source_attribute_id"]
        attr_name = r["attribute"]
        attr_norm = normalize_key(attr_name)

        best_t = None
        best_score = -1

        for t, t_norm in target_norm.items():
            score = fuzz.token_set_ratio(attr_norm, t_norm)

            if "brand" in attr_norm and "brand" in t_norm:
                score += 15
            if "manufacturer" in attr_norm and "brand" in t_norm:
                score += 12
            if "model" in attr_norm and "model" in t_norm:
                score += 15
            if "screen" in attr_norm and "screen" in t_norm:
                score += 10
            if "resolution" in attr_norm and "resolution" in t_norm:
                score += 10
            if "refresh" in attr_norm and "refresh" in t_norm:
                score += 10
            if "response" in attr_norm and "response" in t_norm:
                score += 10

            score = min(score, 100)
            if score > best_score:
                best_score = score
                best_t = t

        pred = best_t if best_score >= threshold else "ABSTAIN"
        rows.append({
            "source_attribute_id": source_attr_id,
            "source": r["source"],
            "attribute": attr_name,
            "predicted_target": pred,
            "confidence": round(best_score / 100, 4)
        })

    return pd.DataFrame(rows)

schema_baseline = baseline_schema_align(attr_stats, MEDIATED_SCHEMA)
schema_baseline.to_csv(RESULTS_DIR / "schema_alignment_baseline.csv", index=False)
schema_baseline.head()

,source_attribute_id,source,attribute,predicted_target,confidence
0,source_alpha//cost,source_alpha,cost,ABSTAIN,0.2667
1,source_alpha//currency,source_alpha,currency,priceCurrency,0.7619
2,source_alpha//details,source_alpha,details,ABSTAIN,0.5000
3,source_alpha//manufacturer,source_alpha,manufacturer,ABSTAIN,0.3553
4,source_alpha//product_name,source_alpha,product_name,ABSTAIN,0.3529


In [7]:
def evaluate_schema_alignment(pred_df, gt_df, label="baseline"):
    merged = gt_df.merge(
        pred_df[["source_attribute_id", "predicted_target"]],
        on="source_attribute_id",
        how="left"
    )
    merged["predicted_target"] = merged["predicted_target"].fillna("ABSTAIN")
    merged["correct"] = merged["predicted_target"] == merged["target_attribute_name"]

    accuracy = merged["correct"].mean() if len(merged) else 0.0

    y_true_pairs = set(zip(merged["source_attribute_id"], merged["target_attribute_name"]))
    y_pred_pairs = set(
        zip(
            merged.loc[merged["predicted_target"] != "ABSTAIN", "source_attribute_id"],
            merged.loc[merged["predicted_target"] != "ABSTAIN", "predicted_target"]
        )
    )

    tp = len(y_true_pairs & y_pred_pairs)
    fp = len(y_pred_pairs - y_true_pairs)
    fn = len(y_true_pairs - y_pred_pairs)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "component": "schema_alignment",
        "pipeline": label,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "n_eval": len(merged),
        "tp": tp,
        "fp": fp,
        "fn": fn
    }

schema_baseline_metrics = evaluate_schema_alignment(schema_baseline, schema_eval_gt, "baseline")
schema_baseline_metrics

{'component': 'schema_alignment',
 'pipeline': 'baseline',
 'accuracy': 0.4,
 'precision': 1.0,
 'recall': 0.4,
 'f1': 0.5714285714285715,
 'n_eval': 15,
 'tp': 6,
 'fp': 0,
 'fn': 9}

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading model:", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval()
print("Model loaded.")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading model: Qwen/Qwen2.5-7B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded.


In [9]:
LLM_LOGS = []

@torch.inference_mode()
def generate_llm(prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)
    return text.strip()

def extract_json(text):
    if text is None:
        return None

    text = text.strip()
    text = re.sub(r"^```json", "", text.strip(), flags=re.IGNORECASE).strip()
    text = re.sub(r"^```", "", text.strip()).strip()
    text = re.sub(r"```$", "", text.strip()).strip()

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass

    m = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass

    try:
        return json.loads(text)
    except Exception:
        return None

def log_llm(stage, prompt, raw_response, parsed, success, metadata=None):
    LLM_LOGS.append({
        "stage": stage,
        "prompt": prompt,
        "raw_response": raw_response,
        "parsed": parsed,
        "success": success,
        "metadata": metadata or {}
    })

In [10]:
SCHEMA_ALIGNMENT_PROMPT_TEMPLATE = """
You are doing schema alignment for Big Data Integration.

Mediated schema:
{mediated_schema}

Source name:
{source_name}

Source attributes with sample values:
{attributes_block}

Task:
Map each source attribute to exactly one mediated attribute when you are confident.
If there is no good mapping, use "ABSTAIN".

Return ONLY valid JSON with this format:
{{
  "mappings": [
    {{
      "source_attribute": "original attribute name",
      "mediated_attribute": "one of the mediated schema attributes or ABSTAIN",
      "confidence": 0.0,
      "reason": "short reason"
    }}
  ]
}}

Rules:
- Do not invent mediated attributes.
- Use only attributes from the mediated schema.
- Use ABSTAIN when unsure.
- Return JSON only.
"""

RECORD_LINKAGE_PROMPT_TEMPLATE = """
You are doing entity resolution for product specifications.

Decide if Record A and Record B refer to the same real-world product.

Record A:
{record_a}

Record B:
{record_b}

Return ONLY valid JSON:
{{
  "match": true,
  "confidence": 0.0,
  "reason": "short explanation"
}}

Rules:
- match must be true or false.
- Focus on title, brand, model, screen size, resolution, and other technical details.
- Different models usually mean non-match.
- Do not hallucinate missing information.
- Return JSON only.
"""

DATA_FUSION_PROMPT_TEMPLATE = """
You are doing data fusion for product records.

Choose one canonical value for each attribute.

Mediated schema:
{mediated_schema}

Records:
{records_block}

Return ONLY compact valid JSON with exactly this format:
{{
  "fused": {{
    "title": "short selected value",
    "brand": "short selected value",
    "description": "short selected value",
    "price": "short selected value",
    "priceCurrency": "short selected value"
  }}
}}

Rules:
- Return JSON only.
- Do not include explanations.
- Do not include confidence.
- Do not invent unsupported values.
- Copy each selected value exactly from one of the provided records.
- Do not shorten, combine, translate, infer, or invent values.
- Use an empty string if no value is supported.
"""

(PROMPTS_DIR / "schema_alignment_prompt.txt").write_text(SCHEMA_ALIGNMENT_PROMPT_TEMPLATE)
(PROMPTS_DIR / "record_linkage_prompt.txt").write_text(RECORD_LINKAGE_PROMPT_TEMPLATE)
(PROMPTS_DIR / "data_fusion_prompt.txt").write_text(DATA_FUSION_PROMPT_TEMPLATE)

743

In [11]:
def build_attributes_block(source_attr_stats, max_attrs=35):
    lines = []
    source_attr_stats = source_attr_stats.sort_values("count", ascending=False).head(max_attrs)

    for _, r in source_attr_stats.iterrows():
        samples = r["samples"]
        if isinstance(samples, str):
            samples = [samples]
        samples_str = "; ".join([str(s)[:80] for s in samples[:3]])
        lines.append(f"- {r['attribute']} | count={r['count']} | samples: {samples_str}")

    return "\n".join(lines)

def llm_schema_align_for_source(source_name, source_attr_stats, mediated_schema):
    attributes_block = build_attributes_block(source_attr_stats, MAX_ATTRS_PER_SOURCE_FOR_LLM)

    prompt = SCHEMA_ALIGNMENT_PROMPT_TEMPLATE.format(
        mediated_schema=json.dumps(mediated_schema, indent=2),
        source_name=source_name,
        attributes_block=attributes_block
    )

    raw = generate_llm(prompt, max_new_tokens=768)
    parsed = extract_json(raw)

    success = isinstance(parsed, dict) and "mappings" in parsed and isinstance(parsed["mappings"], list)
    log_llm(
        stage="schema_alignment",
        prompt=prompt,
        raw_response=raw,
        parsed=parsed,
        success=success,
        metadata={"source": source_name}
    )

    rows = []
    if success:
        for m in parsed["mappings"]:
            src_attr = str(m.get("source_attribute", "")).strip()
            pred = str(m.get("mediated_attribute", "ABSTAIN")).strip()
            conf = m.get("confidence", 0.0)
            reason = str(m.get("reason", ""))

            if pred not in mediated_schema:
                pred = "ABSTAIN"

            source_attribute_id = f"{source_name}//{src_attr}"
            rows.append({
                "source": source_name,
                "attribute": src_attr,
                "source_attribute_id": source_attribute_id,
                "predicted_target": pred,
                "confidence": conf,
                "reason": reason
            })

    return pd.DataFrame(rows)

llm_schema_rows = []

for source_name, group in tqdm(attr_stats.groupby("source"), desc="LLM schema alignment by source"):
    df_source = llm_schema_align_for_source(source_name, group, MEDIATED_SCHEMA)
    if len(df_source):
        llm_schema_rows.append(df_source)

schema_llm_raw = pd.concat(llm_schema_rows, ignore_index=True) if llm_schema_rows else pd.DataFrame()

schema_llm = schema_baseline.copy()
schema_llm["reason"] = "baseline fallback"

if len(schema_llm_raw):
    llm_map = schema_llm_raw.set_index("source_attribute_id").to_dict("index")
    for idx, row in schema_llm.iterrows():
        sid = row["source_attribute_id"]
        if sid in llm_map:
            schema_llm.loc[idx, "predicted_target"] = llm_map[sid]["predicted_target"]
            schema_llm.loc[idx, "confidence"] = llm_map[sid]["confidence"]
            schema_llm.loc[idx, "reason"] = llm_map[sid].get("reason", "llm")

schema_llm.to_csv(RESULTS_DIR / "schema_alignment_llm.csv", index=False)

schema_llm_metrics = evaluate_schema_alignment(schema_llm, schema_eval_gt, "llm_assisted")
schema_llm_metrics

LLM schema alignment by source:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarni

{'component': 'schema_alignment',
 'pipeline': 'llm_assisted',
 'accuracy': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'n_eval': 15,
 'tp': 15,
 'fp': 0,
 'fn': 0}

In [12]:
gold_mapping = dict(
    zip(schema_gt["source_attribute_id"], schema_gt["target_attribute_name"])
)

baseline_mapping = dict(
    zip(schema_baseline["source_attribute_id"], schema_baseline["predicted_target"])
)

llm_mapping = dict(
    zip(schema_llm["source_attribute_id"], schema_llm["predicted_target"])
)

def canonicalize_record(row, mapping, mediated_schema):
    source = row["source"]
    attrs = row["attrs"]
    out = {a: "" for a in mediated_schema}

    for k, v in attrs.items():
        sid = f"{source}//{k}"
        target = mapping.get(sid, "ABSTAIN")
        if target in mediated_schema:
            val = normalize_value(v)
            if val:
                if out[target]:
                    if val not in out[target]:
                        out[target] += " | " + val
                else:
                    out[target] = val

    raw_title = ""
    for k, v in attrs.items():
        nk = normalize_key(k)
        if "page title" in nk or nk == "title" or nk == "name" or "product name" in nk or "item title" in nk:
            raw_title = normalize_value(v)
            break

    if not raw_title:
        vals = sorted([str(v) for v in attrs.values()], key=len, reverse=True)
        raw_title = normalize_value(vals[0]) if vals else ""

    raw_brand = ""
    for k, v in attrs.items():
        nk = normalize_key(k)
        if "brand" in nk or "manufacturer" in nk or "maker" in nk:
            raw_brand = normalize_value(v)
            break

    raw_model = ""
    for k, v in attrs.items():
        nk = normalize_key(k)
        if "model" in nk or "part number" in nk or "mpn" in nk:
            raw_model = normalize_value(v)
            break

    out["__title"] = raw_title
    out["__brand"] = raw_brand
    out["__model"] = raw_model

    return out

def add_canonical_columns(df, mapping, prefix):
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Canonicalizing {prefix}"):
        can = canonicalize_record(row, mapping, MEDIATED_SCHEMA)
        flat = {
            "record_id": row["record_id"],
            "spec_id": row["spec_id"],
            "entity_id": row["entity_id"],
            "source": row["source"],
        }
        for k, v in can.items():
            flat[f"{prefix}_{k}"] = v
        rows.append(flat)
    return pd.DataFrame(rows)

canon_baseline = add_canonical_columns(work_records, baseline_mapping, "base")
canon_llm = add_canonical_columns(work_records, llm_mapping, "llm")
canon_gold = add_canonical_columns(work_records, gold_mapping, "gold")

canon_baseline.to_csv(PROCESSED_DIR / "canonical_baseline.csv", index=False)
canon_llm.to_csv(PROCESSED_DIR / "canonical_llm.csv", index=False)
canon_gold.to_csv(PROCESSED_DIR / "canonical_gold.csv", index=False)

canon_baseline.head()

Canonicalizing base:   0%|          | 0/1800 [00:00<?, ?it/s]

Canonicalizing llm:   0%|          | 0/1800 [00:00<?, ?it/s]

Canonicalizing gold:   0%|          | 0/1800 [00:00<?, ?it/s]

,record_id,spec_id,entity_id,source,base_title,base_brand,base_description,base_price,base_priceCurrency,base___title,base___brand,base___model
0,r0,source_alpha//27282600,1001446,source_alpha,,,,,cad,apple lightning digital av adapter - lightning...,,
1,r1,source_beta//3658626,1001446,source_beta,,,use the lightning digital av adapter with your...,,nzd,lightning to digital av adapter white,,
2,r2,source_gamma//45955817,1001446,source_gamma,apple lightning digital av adaptor,,,75.00,,apple lightning digital av adaptor,apple,
3,r3,source_alpha//58093698,1001446,source_alpha,,,,,aud,apple lightning digital av adapter,,
4,r4,source_beta//6750484,1001446,source_beta,,apple,use the lightning digital av adapter with your...,,usd,apple lightning to digital av adapter,apple,


In [13]:
def record_view(canon_row, prefix):
    d = {}
    for a in MEDIATED_SCHEMA:
        d[a] = canon_row.get(f"{prefix}_{a}", "")
    d["title"] = canon_row.get(f"{prefix}___title", "")
    d["brand"] = canon_row.get(f"{prefix}___brand", "")
    d["model"] = canon_row.get(f"{prefix}___model", "")
    return d

def get_block_keys(row, prefix):
    title = row.get(f"{prefix}___title", "")
    brand = row.get(f"{prefix}___brand", "")
    model = row.get(f"{prefix}___model", "")

    keys = set()

    brand_n = normalize_key(brand)
    model_n = normalize_key(model)

    stop = {
        "the", "and", "for", "with", "from", "new",
        "black", "white", "red", "blue", "green",
        "inch", "inches", "pack", "set", "one",
        "free", "shipping", "original", "genuine",
        "brand", "official", "sale", "usb", "cable",
        "product", "compatible", "case", "cover"
    }

    toks = [
        t for t in tokenize(title)
        if len(t) > 2 and t not in stop
    ]

    if brand_n and toks:
        keys.add(f"brand_title::{brand_n[:20]}::{toks[0]}")

    if model_n:
        keys.add(f"model::{model_n[:30]}")

    for t in toks[:4]:
        keys.add(f"title_token::{t}")

    for i in range(min(len(toks) - 1, 3)):
        keys.add(f"title_bigram::{toks[i]}::{toks[i+1]}")

    if brand_n:
        for t in toks[:4]:
            keys.add(f"brand_token::{brand_n[:20]}::{t}")

    if len(toks) >= 2:
        keys.add(f"title2::{toks[0]}::{toks[1]}")

    return keys

def generate_candidate_pairs(canon_df, prefix, max_block_size=250):
    blocks = defaultdict(list)

    for _, row in canon_df.iterrows():
        rid = row["record_id"]
        for k in get_block_keys(row, prefix):
            blocks[k].append(rid)

    rid_to_source = dict(zip(canon_df["record_id"], canon_df["source"]))
    pairs = set()
    block_stats = []

    for key, ids in blocks.items():
        ids = list(set(ids))
        if len(ids) < 2:
            continue

        block_stats.append({"block_key": key, "block_size": len(ids)})

        if len(ids) > max_block_size:
            continue

        for a, b in itertools.combinations(sorted(ids), 2):
            if rid_to_source[a] == rid_to_source[b]:
                continue
            pairs.add(tuple(sorted((a, b))))

    pairs_df = pd.DataFrame(list(pairs), columns=["record_id_1", "record_id_2"])
    block_stats_df = pd.DataFrame(block_stats).sort_values("block_size", ascending=False)

    return pairs_df, block_stats_df

candidate_pairs_baseline, block_stats_baseline = generate_candidate_pairs(canon_baseline, "base")
candidate_pairs_llm, block_stats_llm = generate_candidate_pairs(canon_llm, "llm")

candidate_pairs_baseline.to_csv(RESULTS_DIR / "candidate_pairs_baseline.csv", index=False)
candidate_pairs_llm.to_csv(RESULTS_DIR / "candidate_pairs_llm.csv", index=False)
block_stats_baseline.to_csv(RESULTS_DIR / "blocking_stats_baseline.csv", index=False)
block_stats_llm.to_csv(RESULTS_DIR / "blocking_stats_llm.csv", index=False)

print("Baseline candidate pairs:", len(candidate_pairs_baseline))
print("LLM candidate pairs:", len(candidate_pairs_llm))
print("Top block sizes:")
block_stats_llm.head()

Baseline candidate pairs: 33855
LLM candidate pairs: 33855
Top block sizes:


,block_key,block_size
283,title_token::link,100
120,title_token::epson,94
52,title_token::asus,84
14,title_token::shimano,70
151,title_token::kingston,68


In [14]:
def build_gold_pairs(records_df):
    gold_pairs = set()
    rid_to_source = dict(zip(records_df["record_id"], records_df["source"]))

    for entity_id, g in records_df.groupby("entity_id"):
        ids = sorted(g["record_id"].tolist())

        if len(ids) >= 2:
            for a, b in itertools.combinations(ids, 2):
                if rid_to_source[a] != rid_to_source[b]:
                    gold_pairs.add(tuple(sorted((a, b))))

    return gold_pairs

gold_pairs = build_gold_pairs(work_records)

all_possible_cross_source = 0
sources_by_rid = dict(zip(work_records["record_id"], work_records["source"]))
all_ids = work_records["record_id"].tolist()

for a, b in itertools.combinations(all_ids, 2):
    if sources_by_rid[a] != sources_by_rid[b]:
        all_possible_cross_source += 1

print("Gold positive pairs:", len(gold_pairs))
print("All possible cross-source pairs:", all_possible_cross_source)

if len(gold_pairs) < 300:
    print("WARNING: < 300 positive pairs. Consider larger subset.")

Gold positive pairs: 3600
All possible cross-source pairs: 1080000


In [15]:
def evaluate_linkage(linkage_df, gold_pairs, label="baseline"):
    pred_pairs = set()
    for _, r in linkage_df[linkage_df["pred_match"] == 1].iterrows():
        pred_pairs.add(tuple(sorted((r["record_id_1"], r["record_id_2"]))))

    tp = len(pred_pairs & gold_pairs)
    fp = len(pred_pairs - gold_pairs)
    fn = len(gold_pairs - pred_pairs)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "component": "record_linkage",
        "pipeline": label,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "candidate_pairs": len(linkage_df),
        "predicted_matches": len(pred_pairs),
        "gold_positive_pairs": len(gold_pairs),
        "tp": tp,
        "fp": fp,
        "fn": fn
    }

In [16]:
def make_record_dicts(canon_df, prefix):
    out = {}

    for _, row in canon_df.iterrows():
        d = {
            "record_id": row["record_id"],
            "spec_id": row["spec_id"],
            "entity_id": row["entity_id"],
            "source": row["source"],
        }

        for a in MEDIATED_SCHEMA:
            v = row.get(f"{prefix}_{a}", "")
            if pd.isna(v):
                v = ""
            d[a] = v

        for aux in ["title", "brand", "model"]:
            v = row.get(f"{prefix}___{aux}", "")
            if not pd.isna(v) and str(v).strip():
                d[aux] = v

        out[row["record_id"]] = d

    return out


def pair_features(a, b):
    title_sim = sim_text(a.get("title", ""), b.get("title", ""))
    brand_sim = exact_or_sim(a.get("brand", ""), b.get("brand", ""))
    model_sim = exact_or_sim(a.get("model", ""), b.get("model", ""))

    mediated_sims = []
    for attr in MEDIATED_SCHEMA:
        va = a.get(attr, "")
        vb = b.get(attr, "")
        if va and vb:
            mediated_sims.append(sim_text(va, vb))

    attr_sim = float(np.mean(mediated_sims)) if mediated_sims else 0.0

    score = (
        0.45 * title_sim +
        0.20 * brand_sim +
        0.20 * model_sim +
        0.15 * attr_sim
    )

    return {
        "title_sim": title_sim,
        "brand_sim": brand_sim,
        "model_sim": model_sim,
        "attr_sim": attr_sim,
        "score": score
    }


def baseline_pair_matching(candidate_pairs, record_dict):
    rows = []

    for _, r in tqdm(candidate_pairs.iterrows(), total=len(candidate_pairs), desc="Baseline pair matching"):
        r1, r2 = r["record_id_1"], r["record_id_2"]

        a = record_dict[r1]
        b = record_dict[r2]

        feats = pair_features(a, b)
        pred = feats["score"] >= MATCH_THRESHOLD

        rows.append({
            "record_id_1": r1,
            "record_id_2": r2,
            **feats,
            "pred_match": int(pred)
        })

    return pd.DataFrame(rows)

In [17]:
base_records = make_record_dicts(canon_baseline, "base")
llm_records = make_record_dicts(canon_llm, "llm")

linkage_baseline = baseline_pair_matching(candidate_pairs_baseline, base_records)
linkage_baseline.to_csv(RESULTS_DIR / "record_linkage_baseline.csv", index=False)

linkage_baseline_metrics = evaluate_linkage(linkage_baseline, gold_pairs, "baseline")
linkage_baseline_metrics

Baseline pair matching:   0%|          | 0/33855 [00:00<?, ?it/s]

{'component': 'record_linkage',
 'pipeline': 'baseline',
 'precision': 0.30460067200827085,
 'recall': 0.6547222222222222,
 'f1': 0.415769977068266,
 'candidate_pairs': 33855,
 'predicted_matches': 7738,
 'gold_positive_pairs': 3600,
 'tp': 2357,
 'fp': 5381,
 'fn': 1243}

In [18]:
def compact_record_for_prompt(record):
    keep = list(dict.fromkeys(["title", "brand", "model"] + MEDIATED_SCHEMA))
    parts = []

    for k in keep:
        v = record.get(k, "")
        if v:
            parts.append(f"{k}: {str(v)[:250]}")

    return "\n".join(parts)

def llm_judge_pair(record_a, record_b, metadata=None):
    prompt = RECORD_LINKAGE_PROMPT_TEMPLATE.format(
        record_a=compact_record_for_prompt(record_a),
        record_b=compact_record_for_prompt(record_b)
    )

    raw = generate_llm(prompt, max_new_tokens=240)
    parsed = extract_json(raw)

    success = isinstance(parsed, dict) and "match" in parsed
    log_llm(
        stage="record_linkage",
        prompt=prompt,
        raw_response=raw,
        parsed=parsed,
        success=success,
        metadata=metadata
    )

    if not success:
        return None

    match = parsed.get("match", False)
    confidence = parsed.get("confidence", 0.0)
    reason = parsed.get("reason", "")

    return {
        "llm_match": bool(match),
        "llm_confidence": float(confidence) if isinstance(confidence, (int, float)) else 0.0,
        "llm_reason": str(reason)
    }

linkage_llm_base_scores = baseline_pair_matching(candidate_pairs_llm, llm_records)

borderline = linkage_llm_base_scores[
    (linkage_llm_base_scores["score"] >= BORDERLINE_LOW) &
    (linkage_llm_base_scores["score"] < BORDERLINE_HIGH)
].copy()

borderline = borderline.sort_values("score", ascending=False).head(MAX_LLM_LINKAGE_CALLS)
borderline_ids = set(zip(borderline["record_id_1"], borderline["record_id_2"]))

print("Borderline pairs selected for LLM:", len(borderline_ids))

llm_decisions = {}

for _, r in tqdm(borderline.iterrows(), total=len(borderline), desc="LLM record linkage"):
    rid1, rid2 = r["record_id_1"], r["record_id_2"]
    result = llm_judge_pair(
        llm_records[rid1],
        llm_records[rid2],
        metadata={"record_id_1": rid1, "record_id_2": rid2, "score": r["score"]}
    )
    if result is not None:
        llm_decisions[(rid1, rid2)] = result

linkage_llm = linkage_llm_base_scores.copy()
linkage_llm["used_llm"] = 0
linkage_llm["llm_confidence"] = np.nan
linkage_llm["llm_reason"] = ""

for idx, r in linkage_llm.iterrows():
    key = (r["record_id_1"], r["record_id_2"])
    if key in llm_decisions:
        d = llm_decisions[key]
        linkage_llm.loc[idx, "pred_match"] = int(d["llm_match"])
        linkage_llm.loc[idx, "used_llm"] = 1
        linkage_llm.loc[idx, "llm_confidence"] = d["llm_confidence"]
        linkage_llm.loc[idx, "llm_reason"] = d["llm_reason"]

linkage_llm.to_csv(RESULTS_DIR / "record_linkage_llm.csv", index=False)

linkage_llm_metrics = evaluate_linkage(linkage_llm, gold_pairs, "llm_assisted")
linkage_llm_metrics

Baseline pair matching:   0%|          | 0/33855 [00:00<?, ?it/s]

Borderline pairs selected for LLM: 300


LLM record linkage:   0%|          | 0/300 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{'component': 'record_linkage',
 'pipeline': 'llm_assisted',
 'precision': 0.31755002291125706,
 'recall': 0.5775,
 'f1': 0.4097762885581946,
 'candidate_pairs': 33855,
 'predicted_matches': 6547,
 'gold_positive_pairs': 3600,
 'tp': 2079,
 'fp': 4468,
 'fn': 1521}

In [19]:
class UnionFind:
    def __init__(self, items):
        self.parent = {x: x for x in items}
        self.rank = {x: 0 for x in items}

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1

def build_clusters(records_df, linkage_df):
    uf = UnionFind(records_df["record_id"].tolist())

    for _, r in linkage_df[linkage_df["pred_match"] == 1].iterrows():
        uf.union(r["record_id_1"], r["record_id_2"])

    root_to_cluster = {}
    rows = []
    for rid in records_df["record_id"]:
        root = uf.find(rid)
        if root not in root_to_cluster:
            root_to_cluster[root] = f"c{len(root_to_cluster)}"
        rows.append({
            "record_id": rid,
            "cluster_id": root_to_cluster[root]
        })

    return pd.DataFrame(rows)

clusters_baseline = build_clusters(work_records, linkage_baseline)
clusters_llm = build_clusters(work_records, linkage_llm)

clusters_baseline.to_csv(RESULTS_DIR / "clusters_baseline.csv", index=False)
clusters_llm.to_csv(RESULTS_DIR / "clusters_llm.csv", index=False)

print("Baseline clusters:", clusters_baseline["cluster_id"].nunique())
print("LLM clusters:", clusters_llm["cluster_id"].nunique())

Baseline clusters: 140
LLM clusters: 197


In [20]:
def choose_majority_value(values):
    values = [normalize_value(v) for v in values if normalize_value(v)]
    if not values:
        return ""

    counts = Counter(values)
    max_count = max(counts.values())
    candidates = [v for v, c in counts.items() if c == max_count]

    candidates = sorted(candidates, key=lambda x: (len(x), x), reverse=True)
    return candidates[0]

def build_gold_canonical_by_entity(canon_gold_df):
    rows = []
    for entity_id, g in canon_gold_df.groupby("entity_id"):
        row = {"entity_id": entity_id}
        for attr in MEDIATED_SCHEMA:
            vals = g[f"gold_{attr}"].dropna().astype(str).tolist()
            vals = [v for v in vals if normalize_value(v)]
            row[attr] = choose_majority_value(vals)

            unique_vals = set(normalize_value(v) for v in vals if normalize_value(v))
            row[f"{attr}__n_values"] = len(unique_vals)
            row[f"{attr}__has_conflict"] = int(len(unique_vals) > 1)
        rows.append(row)
    return pd.DataFrame(rows)

fusion_gold = build_gold_canonical_by_entity(canon_gold)
fusion_gold.to_csv(PROCESSED_DIR / "fusion_silver_ground_truth.csv", index=False)

conflict_count = int(fusion_gold[[f"{a}__has_conflict" for a in MEDIATED_SCHEMA]].sum().sum())
print("Fusion silver truth entities:", fusion_gold.shape)
print("Conflicting entity-attribute cases:", conflict_count)

if conflict_count < 100:
    print("WARNING: < 100 conflicts detected in selected mediated schema. Consider adding more attributes.")

Fusion silver truth entities: (300, 16)
Conflicting entity-attribute cases: 1220


In [21]:
def fuse_cluster(records_ids, canon_df, prefix):
    sub = canon_df[canon_df["record_id"].isin(records_ids)]
    fused = {}
    for attr in MEDIATED_SCHEMA:
        vals = sub[f"{prefix}_{attr}"].dropna().astype(str).tolist()
        vals = [v for v in vals if normalize_value(v)]
        fused[attr] = choose_majority_value(vals)
    return fused

def build_fused_dataset(canon_df, clusters_df, prefix, pipeline_name):
    rows = []
    cluster_map = clusters_df.groupby("cluster_id")["record_id"].apply(list).to_dict()

    for cid, rids in cluster_map.items():
        fused = fuse_cluster(rids, canon_df, prefix)
        row = {
            "cluster_id": cid,
            "pipeline": pipeline_name,
            "n_records": len(rids),
            "record_ids": json.dumps(rids)
        }
        row.update(fused)
        rows.append(row)

    return pd.DataFrame(rows)

fused_baseline = build_fused_dataset(canon_baseline, clusters_baseline, "base", "baseline")
fused_llm_initial = build_fused_dataset(canon_llm, clusters_llm, "llm", "llm_assisted_initial")

fused_baseline.to_csv(RESULTS_DIR / "integrated_dataset_baseline.csv", index=False)
fused_llm_initial.to_csv(RESULTS_DIR / "integrated_dataset_llm_initial.csv", index=False)

fused_baseline.head()

,cluster_id,pipeline,n_records,record_ids,title,brand,description,price,priceCurrency
0,c0,baseline,47,"[""r0"", ""r1"", ""r2"", ""r3"", ""r4"", ""r5"", ""r66"", ""r...",new genuine apple macbook pro a1278 a1280 mags...,apple,"summary:- designed by apple- automatically on,...",159.00,usd
1,c1,baseline,119,"[""r6"", ""r8"", ""r9"", ""r10"", ""r11"", ""r30"", ""r31"",...",sram disc brake pad set organic with steel bac...,shimano,the bb-mt800-pa bottom bracket by shimano - pr...,1.4999e2,usd
2,c10,baseline,1,"[""r57""]",,,,,gbp
3,c100,baseline,1,"[""r1096""]",,,mpn: sdsqxa1-256g-gn6mavendor:sandisksandisk e...,,aud
4,c101,baseline,1,"[""r1121""]",es-16-150wubiquiti 16 port poe edgeswitch 150 ...,,,1.98,


In [22]:
def records_block_for_fusion(record_ids, canon_df, prefix, max_records=3):
    sub = canon_df[canon_df["record_id"].isin(record_ids)].head(max_records)
    blocks = []
    for _, row in sub.iterrows():
        d = {
            "record_id": row["record_id"],
            "source": row["source"]
        }
        for a in MEDIATED_SCHEMA:
            val = row.get(f"{prefix}_{a}", "")
            if val:
                d[a] = val
        title = row.get(f"{prefix}___title", "")
        brand = row.get(f"{prefix}___brand", "")
        model_v = row.get(f"{prefix}___model", "")
        if title:
            d["raw_title"] = title
        if brand:
            d["raw_brand"] = brand
        if model_v:
            d["raw_model"] = model_v
        blocks.append(d)
    return json.dumps(blocks, indent=2)

def has_cluster_conflict(record_ids, canon_df, prefix):
    sub = canon_df[canon_df["record_id"].isin(record_ids)]
    for attr in MEDIATED_SCHEMA:
        vals = set(normalize_value(v) for v in sub[f"{prefix}_{attr}"].dropna().astype(str) if normalize_value(v))
        if len(vals) > 1:
            return True
    return False

def llm_fuse_cluster(record_ids, canon_df, prefix, metadata=None):
    records_block = records_block_for_fusion(
        record_ids,
        canon_df,
        prefix
    )

    prompt = DATA_FUSION_PROMPT_TEMPLATE.format(
        mediated_schema=json.dumps(MEDIATED_SCHEMA, indent=2),
        records_block=records_block
    )

    raw = generate_llm(prompt, max_new_tokens=384)
    parsed = extract_json(raw)

    valid_json = (
        isinstance(parsed, dict)
        and isinstance(parsed.get("fused"), dict)
    )

    fallback = fuse_cluster(
        record_ids,
        canon_df,
        prefix
    )

    if not valid_json:
        log_llm(
            stage="data_fusion",
            prompt=prompt,
            raw_response=raw,
            parsed=parsed,
            success=False,
            metadata={
                **(metadata or {}),
                "fallback": "invalid_json"
            }
        )

        return fallback, False

    sub = canon_df[
        canon_df["record_id"].isin(record_ids)
    ]

    fused = fallback.copy()
    accepted_attributes = []
    rejected_attributes = []

    for attr in MEDIATED_SCHEMA:
        val_obj = parsed["fused"].get(attr, "")

        if isinstance(val_obj, dict):
            candidate = normalize_value(
                val_obj.get("value", "")
            )
        else:
            candidate = normalize_value(val_obj)

        supported_values = {
            normalize_value(value)
            for value in sub[
                f"{prefix}_{attr}"
            ].dropna().astype(str)
            if normalize_value(value)
        }

        if candidate and candidate in supported_values:
            fused[attr] = candidate
            accepted_attributes.append(attr)

        elif candidate:
            rejected_attributes.append(attr)

    log_llm(
        stage="data_fusion",
        prompt=prompt,
        raw_response=raw,
        parsed=parsed,
        success=(
            bool(accepted_attributes)
            and not rejected_attributes
        ),
        metadata={
            **(metadata or {}),
            "accepted_attributes": accepted_attributes,
            "unsupported_attributes": rejected_attributes,
            "fallback": "majority_voting"
        }
    )

    return fused, bool(accepted_attributes)


fused_llm = fused_llm_initial.copy()
fused_llm["used_llm_fusion"] = 0

cluster_to_records = (
    clusters_llm
    .groupby("cluster_id")["record_id"]
    .apply(list)
    .to_dict()
)

conflict_clusters = []

for cid, rids in cluster_to_records.items():
    if (
        len(rids) >= 2
        and has_cluster_conflict(
            rids,
            canon_llm,
            "llm"
        )
    ):
        conflict_clusters.append((cid, rids))

conflict_clusters = conflict_clusters[
    :MAX_LLM_FUSION_CALLS
]

print(
    "Conflict clusters selected for LLM fusion:",
    len(conflict_clusters)
)

for cid, rids in tqdm(
    conflict_clusters,
    desc="LLM data fusion"
):
    result, used_valid_llm_value = llm_fuse_cluster(
        rids,
        canon_llm,
        "llm",
        metadata={
            "cluster_id": cid,
            "record_ids": rids
        }
    )

    idxs = fused_llm.index[
        fused_llm["cluster_id"] == cid
    ].tolist()

    if not idxs:
        continue

    idx = idxs[0]

    for attr, val in result.items():
        if attr in MEDIATED_SCHEMA and val:
            fused_llm.loc[idx, attr] = val

    fused_llm.loc[
        idx,
        "used_llm_fusion"
    ] = int(used_valid_llm_value)

fused_llm["pipeline"] = "llm_assisted"

fused_llm.to_csv(
    RESULTS_DIR / "integrated_dataset_llm.csv",
    index=False
)

fused_llm.head()

Conflict clusters selected for LLM fusion: 50


LLM data fusion:   0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarni

,cluster_id,pipeline,n_records,record_ids,title,brand,description,price,priceCurrency,used_llm_fusion
0,c0,llm_assisted,46,"[""r0"", ""r1"", ""r2"", ""r3"", ""r4"", ""r5"", ""r66"", ""r...",apple lightning digital av adapter - lightning...,apple,use the lightning digital av adapter with your...,75.00,aud,1
1,c1,llm_assisted,111,"[""r6"", ""r30"", ""r31"", ""r32"", ""r33"", ""r34"", ""r35...",shimano cn-hg95 10-speed hg-x chain - 116 links,shimano,the bb-mt800-pa bottom bracket by shimano - pr...,29.99,gbp,1
2,c10,llm_assisted,36,"[""r42"", ""r43"", ""r44"", ""r45"", ""r46"", ""r47"", ""r3...",jabra biz 2300 duo usb ms lync headset,jabra,your contact center agents are your brand amba...,149.00,usd,1
3,c100,llm_assisted,1,"[""r713""]",intel i9 9900 3.1ghz 8 core cpu | discount com...,,view and shop for intel i9 9900 3.1ghz 8 core ...,718.95,aud,0
4,c101,llm_assisted,6,"[""r714"", ""r715"", ""r716"", ""r717"", ""r718"", ""r719""]",pokemon sword & shield rebel clash booster,nintendo,pokémon tcg: sword & shield—rebel clash booster,4.5,eur,1


In [23]:
def map_pred_clusters_to_gold(clusters_df, records_df):
    tmp = clusters_df.merge(records_df[["record_id", "entity_id"]], on="record_id", how="left")
    mapping = {}

    for cid, g in tmp.groupby("cluster_id"):
        if g["entity_id"].isna().all():
            continue
        majority_entity = g["entity_id"].value_counts().idxmax()
        mapping[cid] = majority_entity

    return mapping

def evaluate_fusion(fused_df, clusters_df, fusion_gold_df, records_df, label):
    cluster_to_entity = map_pred_clusters_to_gold(clusters_df, records_df)
    gold_by_entity = fusion_gold_df.set_index("entity_id").to_dict("index")

    rows = []
    total = 0
    correct = 0

    for _, row in fused_df.iterrows():
        cid = row["cluster_id"]
        entity = cluster_to_entity.get(cid)
        if entity not in gold_by_entity:
            continue

        gold = gold_by_entity[entity]
        for attr in MEDIATED_SCHEMA:
            gold_val = normalize_value(gold.get(attr, ""))
            pred_val = normalize_value(row.get(attr, ""))

            if not gold_val:
                continue

            is_correct = pred_val == gold_val
            total += 1
            correct += int(is_correct)
            rows.append({
                "pipeline": label,
                "cluster_id": cid,
                "entity_id": entity,
                "attribute": attr,
                "predicted_value": pred_val,
                "gold_value": gold_val,
                "correct": int(is_correct)
            })

    acc = correct / total if total else 0.0
    return {
        "component": "data_fusion",
        "pipeline": label,
        "accuracy": acc,
        "n_eval_values": total,
        "correct_values": correct
    }, pd.DataFrame(rows)

fusion_baseline_metrics, fusion_baseline_eval = evaluate_fusion(
    fused_baseline, clusters_baseline, fusion_gold, work_records, "baseline"
)

fusion_llm_metrics, fusion_llm_eval = evaluate_fusion(
    fused_llm, clusters_llm, fusion_gold, work_records, "llm_assisted"
)

fusion_baseline_eval.to_csv(RESULTS_DIR / "fusion_eval_baseline.csv", index=False)
fusion_llm_eval.to_csv(RESULTS_DIR / "fusion_eval_llm.csv", index=False)

fusion_baseline_metrics, fusion_llm_metrics

({'component': 'data_fusion',
  'pipeline': 'baseline',
  'accuracy': 0.20029455081001474,
  'n_eval_values': 679,
  'correct_values': 136},
 {'component': 'data_fusion',
  'pipeline': 'llm_assisted',
  'accuracy': 0.23903966597077245,
  'n_eval_values': 958,
  'correct_values': 229})

In [24]:
def build_linkage_error_examples(linkage_df, record_dict, gold_pairs, label, max_examples=20):
    rows = []

    for _, r in linkage_df.iterrows():
        pair = tuple(sorted((r["record_id_1"], r["record_id_2"])))
        pred = int(r["pred_match"])
        gold = int(pair in gold_pairs)

        if pred != gold:
            a = record_dict[r["record_id_1"]]
            b = record_dict[r["record_id_2"]]
            rows.append({
                "pipeline": label,
                "error_type": "false_positive" if pred == 1 else "false_negative",
                "record_id_1": r["record_id_1"],
                "record_id_2": r["record_id_2"],
                "score": r.get("score", None),
                "pred_match": pred,
                "gold_match": gold,
                "record_1": json.dumps(a, ensure_ascii=False),
                "record_2": json.dumps(b, ensure_ascii=False)
            })

    return pd.DataFrame(rows).head(max_examples)

baseline_linkage_errors = build_linkage_error_examples(linkage_baseline, base_records, gold_pairs, "baseline")
llm_linkage_errors = build_linkage_error_examples(linkage_llm, llm_records, gold_pairs, "llm_assisted")

baseline_linkage_errors.to_csv(RESULTS_DIR / "error_examples_linkage_baseline.csv", index=False)
llm_linkage_errors.to_csv(RESULTS_DIR / "error_examples_linkage_llm.csv", index=False)

def fusion_error_examples(fusion_eval_df, label, max_examples=20):
    return fusion_eval_df[fusion_eval_df["correct"] == 0].head(max_examples).assign(pipeline=label)

fusion_errors_baseline = fusion_error_examples(fusion_baseline_eval, "baseline")
fusion_errors_llm = fusion_error_examples(fusion_llm_eval, "llm_assisted")

fusion_errors_baseline.to_csv(RESULTS_DIR / "error_examples_fusion_baseline.csv", index=False)
fusion_errors_llm.to_csv(RESULTS_DIR / "error_examples_fusion_llm.csv", index=False)

baseline_linkage_errors.head()

,pipeline,error_type,record_id_1,record_id_2,score,pred_match,gold_match,record_1,record_2
0,baseline,false_positive,r1066,r723,0.526190,1,0,"{""record_id"": ""r1066"", ""spec_id"": ""source_beta...","{""record_id"": ""r723"", ""spec_id"": ""source_alpha..."
1,baseline,false_positive,r1661,r702,0.501471,1,0,"{""record_id"": ""r1661"", ""spec_id"": ""source_gamm...","{""record_id"": ""r702"", ""spec_id"": ""source_alpha..."
2,baseline,false_positive,r1652,r639,0.533197,1,0,"{""record_id"": ""r1652"", ""spec_id"": ""source_gamm...","{""record_id"": ""r639"", ""spec_id"": ""source_alpha..."
3,baseline,false_positive,r1051,r429,0.432609,1,0,"{""record_id"": ""r1051"", ""spec_id"": ""source_beta...","{""record_id"": ""r429"", ""spec_id"": ""source_alpha..."
4,baseline,false_positive,r430,r462,0.462121,1,0,"{""record_id"": ""r430"", ""spec_id"": ""source_beta/...","{""record_id"": ""r462"", ""spec_id"": ""source_alpha..."


In [25]:
logs_path = RESULTS_DIR / "llm_logs.jsonl"
with open(logs_path, "w", encoding="utf-8") as f:
    for item in LLM_LOGS:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved LLM logs:", logs_path)
print("Total LLM calls:", len(LLM_LOGS))
print(pd.Series([x["stage"] for x in LLM_LOGS]).value_counts())

Saved LLM logs: /content/llm_bdi_project/results/llm_logs.jsonl
Total LLM calls: 353
record_linkage      300
data_fusion          50
schema_alignment      3
Name: count, dtype: int64


In [26]:
metrics_rows = []

metrics_rows.append(schema_baseline_metrics)
metrics_rows.append(schema_llm_metrics)
metrics_rows.append(linkage_baseline_metrics)
metrics_rows.append(linkage_llm_metrics)
metrics_rows.append(fusion_baseline_metrics)
metrics_rows.append(fusion_llm_metrics)

metrics_rows.append({
    "component": "blocking",
    "pipeline": "baseline",
    "candidate_pairs": len(candidate_pairs_baseline),
    "all_possible_cross_source_pairs": all_possible_cross_source,
    "reduction_ratio": 1 - (len(candidate_pairs_baseline) / all_possible_cross_source if all_possible_cross_source else 0)
})

metrics_rows.append({
    "component": "blocking",
    "pipeline": "llm_assisted",
    "candidate_pairs": len(candidate_pairs_llm),
    "all_possible_cross_source_pairs": all_possible_cross_source,
    "reduction_ratio": 1 - (len(candidate_pairs_llm) / all_possible_cross_source if all_possible_cross_source else 0)
})

metrics_rows.append({
    "component": "llm_usage",
    "pipeline": "llm_assisted",
    "total_llm_calls": len(LLM_LOGS),
    "schema_alignment_calls": sum(1 for x in LLM_LOGS if x["stage"] == "schema_alignment"),
    "record_linkage_calls": sum(1 for x in LLM_LOGS if x["stage"] == "record_linkage"),
    "data_fusion_calls": sum(1 for x in LLM_LOGS if x["stage"] == "data_fusion")
})

metrics_summary = pd.DataFrame(metrics_rows)
metrics_summary.to_csv(RESULTS_DIR / "metrics_summary.csv", index=False)

metrics_summary

,component,pipeline,accuracy,precision,recall,f1,n_eval,tp,fp,fn,...,predicted_matches,gold_positive_pairs,n_eval_values,correct_values,all_possible_cross_source_pairs,reduction_ratio,total_llm_calls,schema_alignment_calls,record_linkage_calls,data_fusion_calls
0,schema_alignment,baseline,0.400000,1.000000,0.400000,0.571429,15.0,6.0,0.0,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,schema_alignment,llm_assisted,1.000000,1.000000,1.000000,1.000000,15.0,15.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,record_linkage,baseline,NaN,0.304601,0.654722,0.415770,NaN,2357.0,5381.0,1243.0,...,7738.0,3600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,record_linkage,llm_assisted,NaN,0.317550,0.577500,0.409776,NaN,2079.0,4468.0,1521.0,...,6547.0,3600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,data_fusion,baseline,0.200295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,679.0,136.0,NaN,NaN,NaN,NaN,NaN,NaN
5,data_fusion,llm_assisted,0.239040,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,958.0,229.0,NaN,NaN,NaN,NaN,NaN,NaN
6,blocking,baseline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1080000.0,0.968653,NaN,NaN,NaN,NaN
7,blocking,llm_assisted,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1080000.0,0.968653,NaN,NaN,NaN,NaN
8,llm_usage,llm_assisted,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,353.0,3.0,300.0,50.0


In [27]:
report_md = f"""
# LLM-Assisted Big Data Integration - Experimental Summary

## Dataset

Dataset: WDC Products Multi from Hugging Face.

The experiment uses product offers from the WDC Products benchmark. The Hugging Face version provides a unified table with product attributes and a cluster identifier. Since the assignment requires three heterogeneous sources, I construct three source views by assigning offers to source_alpha, source_beta, and source_gamma, and by renaming attributes differently in each source.
The sources are synthetically constructed from a single public dataset by distributing bids using a round-robin method. This simplifies the integration pipeline because the underlying values ​​are consistent, and it can underestimate the real difficulties of integrating with truly heterogeneous sources.

The working subset contains:

- Records: {len(work_records)}
- Sources: {work_records['source'].nunique()}
- Gold entities: {work_records['entity_id'].nunique()}
- Gold positive matching pairs: {len(gold_pairs)}
- Mediated schema: {MEDIATED_SCHEMA}
- Fusion conflicts detected in silver truth: {conflict_count}

## Pipelines

### Pipeline A - Traditional Baseline

- Schema alignment based on string similarity and simple attribute-name rules.
- Blocking based on brand, model, and title tokens.
- Pairwise matching using weighted similarity over title, brand, model, and mediated attributes.
- Clustering using Union-Find.
- Data fusion using majority voting and longest-value tie-breaking.

### Pipeline B - LLM-Assisted Integration
The LLM-assisted pipeline uses Qwen/Qwen2.5-7B-Instruct in three stages:

1. Schema alignment for source-to-mediated attribute mappings.
2. Record linkage for borderline candidate pairs.
3. Data fusion on a small selected subset of conflicting clusters.

For computational reasons on Colab, LLM-assisted fusion is limited to MAX_LLM_FUSION_CALLS = 50 conflicting clusters, using short prompts with at most 3 records per cluster and max_new_tokens = 384.
The model is prompted to return JSON only. Invalid JSON, missing fields, empty responses, and unsupported values are logged and treated as failures or fallback cases.

## Results

{metrics_summary.to_markdown(index=False)}

## Output files

The notebook generates:

- schema_alignment_baseline.csv
- schema_alignment_llm.csv
- candidate_pairs_baseline.csv
- candidate_pairs_llm.csv
- record_linkage_baseline.csv
- record_linkage_llm.csv
- clusters_baseline.csv
- clusters_llm.csv
- integrated_dataset_baseline.csv
- integrated_dataset_llm.csv
- fusion_eval_baseline.csv
- fusion_eval_llm.csv
- metrics_summary.csv
- llm_logs.jsonl

## Notes
The silver truth of the merger is constructed using majority voting on the values ​​of the gold cluster, which introduces a structural advantage for the baseline that also uses majority voting. The merger results should be interpreted with this limitation in mind.
"""

(RESULTS_DIR / "technical_report_base.md").write_text(report_md, encoding="utf-8")
print(report_md[:2000])


# LLM-Assisted Big Data Integration - Experimental Summary

## Dataset

Dataset: WDC Products Multi from Hugging Face.

The experiment uses product offers from the WDC Products benchmark. The Hugging Face version provides a unified table with product attributes and a cluster identifier. Since the assignment requires three heterogeneous sources, I construct three source views by assigning offers to source_alpha, source_beta, and source_gamma, and by renaming attributes differently in each source.
The sources are synthetically constructed from a single public dataset by distributing bids using a round-robin method. This simplifies the integration pipeline because the underlying values ​​are consistent, and it can underestimate the real difficulties of integrating with truly heterogeneous sources.

The working subset contains:

- Records: 1800
- Sources: 3
- Gold entities: 300
- Gold positive matching pairs: 3600
- Mediated schema: ['title', 'brand', 'description', 'price', 'priceCurrenc

In [28]:
#SOLO PARA DESCARGAR RESULTADOS, NO INCLUIR A LA HORA DE ENTREGAR

from pathlib import Path
import shutil
import os
from google.colab import files

PROJECT_DIR = Path("/content/llm_bdi_project")
ZIP_BASE = Path("/content/llm_bdi_project_submission")
ZIP_PATH = Path(str(ZIP_BASE) + ".zip")

# Comprobación básica
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"No existe la carpeta del proyecto: {PROJECT_DIR}")

# Eliminar zip anterior si existe
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# Crear un pequeño resumen de contenido antes de comprimir
summary_path = PROJECT_DIR / "submission_manifest.txt"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("LLM-Assisted Big Data Integration - Submission Manifest\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Project folder: {PROJECT_DIR}\n\n")
    f.write("Generated files:\n")

    for root, dirs, filenames in os.walk(PROJECT_DIR):
        for filename in filenames:
            file_path = Path(root) / filename
            rel_path = file_path.relative_to(PROJECT_DIR)
            size_kb = file_path.stat().st_size / 1024
            f.write(f"- {rel_path} ({size_kb:.2f} KB)\n")

print("Creating ZIP...")

# Comprimir carpeta completa
shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=str(PROJECT_DIR.parent),
    base_dir=PROJECT_DIR.name
)

print(f"ZIP created: {ZIP_PATH}")
print(f"Size: {ZIP_PATH.stat().st_size / (1024 * 1024):.2f} MB")

# Descargar automáticamente
files.download(str(ZIP_PATH))

Creating ZIP...
ZIP created: /content/llm_bdi_project_submission.zip
Size: 3.29 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
from pathlib import Path
import zipfile
import os
from google.colab import files

# Ruta del ZIP que ya generaste
ZIP_PATH = Path("/content/llm_bdi_project_submission.zip")

# TXT final consolidado
OUTPUT_TXT = Path("/content/llm_bdi_project_submission_all_in_one.txt")

# Extensiones que sí tiene sentido volcar como texto
TEXT_EXTENSIONS = {
    ".txt", ".md", ".csv", ".json", ".jsonl",
    ".py", ".ipynb", ".yaml", ".yml",
    ".html", ".css", ".js", ".log"
}

# Extensiones binarias que se omiten
BINARY_EXTENSIONS = {
    ".pkl", ".pickle", ".zip", ".png", ".jpg", ".jpeg",
    ".pdf", ".parquet", ".bin", ".pt", ".pth"
}

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"No existe el ZIP: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = sorted(zf.namelist())

    with open(OUTPUT_TXT, "w", encoding="utf-8") as out:
        out.write("LLM-BDI PROJECT - ALL FILES CONSOLIDATED\n")
        out.write("=" * 80 + "\n\n")
        out.write(f"Source ZIP: {ZIP_PATH}\n")
        out.write(f"Number of items in ZIP: {len(names)}\n\n")

        out.write("ZIP CONTENTS\n")
        out.write("-" * 80 + "\n")
        for name in names:
            out.write(f"{name}\n")
        out.write("\n\n")

        for name in names:
            if name.endswith("/"):
                continue

            suffix = Path(name).suffix.lower()

            out.write("\n")
            out.write("=" * 100 + "\n")
            out.write(f"FILE: {name}\n")
            out.write("=" * 100 + "\n\n")

            if suffix in BINARY_EXTENSIONS:
                info = zf.getinfo(name)
                out.write(f"[BINARY FILE OMITTED]\n")
                out.write(f"Size: {info.file_size} bytes\n")
                out.write(f"Extension: {suffix}\n")
                continue

            if suffix not in TEXT_EXTENSIONS:
                info = zf.getinfo(name)
                out.write(f"[UNKNOWN/NON-TEXT FILE OMITTED]\n")
                out.write(f"Size: {info.file_size} bytes\n")
                out.write(f"Extension: {suffix if suffix else 'no extension'}\n")
                continue

            try:
                raw = zf.read(name)

                try:
                    text = raw.decode("utf-8")
                except UnicodeDecodeError:
                    text = raw.decode("latin-1", errors="replace")

                out.write(text)

                if not text.endswith("\n"):
                    out.write("\n")

            except Exception as e:
                out.write(f"[ERROR READING FILE]\n{repr(e)}\n")

print(f"TXT creado: {OUTPUT_TXT}")
print(f"Tamaño: {OUTPUT_TXT.stat().st_size / (1024 * 1024):.2f} MB")

files.download(str(OUTPUT_TXT))

TXT creado: /content/llm_bdi_project_submission_all_in_one.txt
Tamaño: 10.44 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>